# Acetylation prediction — pushing Sn, Sp >= 0.70 and MCC >= 0.40

Same `ProteinTransformer` architecture. Everything changed here is training
protocol, not model structure.

**Why the previous run stalled at MCC 0.389**

On balanced data at a 0.5 boundary, `MCC = Sn + Sp - 1`. MCC 0.3887 means
balanced accuracy 0.694. Under a symmetric ROC, AUC 0.7611 supports a best
symmetric operating point of about 0.693 — the model is already at the ceiling
its ranking quality allows. Reaching Sn = Sp = 0.70 needs **AUC around 0.772**.

**The five changes, in order of expected value**

1. **Per-epoch negative resampling.** The old code drew one fixed negative
   subset and threw away ~205k negatives forever. Here the train bin redraws a
   *different* balanced negative sample every epoch from the same train
   proteins. Every epoch is still exactly 1:1, the test set is untouched, but
   over 80 epochs the model sees most of the negative pool instead of 6% of it.
2. **Train to convergence.** Runs finished in 0.9 min, meaning early stopping
   fired around epoch 6 and the cosine schedule never decayed. Now 80 epochs,
   patience 15, with warmup.
3. **Checkpoint on `min(Sn, Sp)`** instead of MCC. MCC is indifferent between
   Sn 0.75 / Sp 0.65 and a symmetric 0.70 / 0.70. Your target is not, so select
   on the quantity you actually care about. Validation only — the test set is
   never consulted.
4. **Ensemble of 3 inits per seed**, probabilities averaged. Reliably worth
   0.01-0.02 AUC and it symmetrises the operating point.
5. **Dropout 0.3 -> 0.15.** The fast early stop is an underfitting signature,
   not an overfitting one.

Expect roughly 15-30 min per seed, against 0.9 before. That is the cost of
actually training.

## 1. GPU

In [ ]:
import torch, subprocess
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('Runtime -> Change runtime type -> GPU, then rerun.')

PyTorch 2.11.0+cu128 | CUDA True
NVIDIA A100-SXM4-40GB, 42.4 GB


## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/dbPTM_Acetylation'
for f in sorted(os.listdir(DRIVE_DIR)):
    p = os.path.join(DRIVE_DIR, f)
    if os.path.isfile(p):
        print(f'  {f:30s} {os.path.getsize(p)/1e9:8.2f} GB')

Mounted at /content/drive
  acet25_X.npy                      12.92 GB
  acet25_meta.npz                    0.04 GB
  acet25_y.npy                       0.00 GB


## 3. Configuration

In [ ]:
SPLIT_SEEDS = [42, 143, 244, 345, 446, 547, 648, 749, 850, 951]

X_PATH    = os.path.join(DRIVE_DIR, 'acet25_X.npy')
Y_PATH    = os.path.join(DRIVE_DIR, 'acet25_y.npy')
META_PATH = os.path.join(DRIVE_DIR, 'acet25_meta.npz')
OUTDIR    = os.path.join(DRIVE_DIR, 'results_colab_v2')
os.makedirs(OUTDIR, exist_ok=True)

TEST_FRAC, VAL_FRAC = 0.20, 0.15

# architecture — unchanged, except dropout
SEQ_LEN, INPUT_DIM, HIDDEN_DIM = 25, 1024, 512
NHEAD, FF_DIM, NLAYERS = 8, 1024, 3
DROPOUT = 0.15                  # was 0.3

# training
THRESHOLD = 0.5
BATCH     = 128                 # was 64; bigger batch, bigger LR
LR        = 3e-4                # was 1e-4
WD        = 1e-4
EPOCHS    = 80                  # was 30
WARMUP    = 5                   # linear warmup epochs before cosine decay
PATIENCE  = 15                  # was 5

RESAMPLE_NEG = True             # redraw the balanced negative sample each epoch
ENSEMBLE     = 3                # models per seed; probabilities averaged
SELECT_BY    = 'min_sn_sp'      # or 'mcc'

KEYS = ('Sn', 'Sp', 'gap', 'Acc', 'MCC', 'AUC', 'AUPRC')
CM_KEYS = ('TN', 'FP', 'FN', 'TP')
TARGET = {'Sn': 0.70, 'Sp': 0.70, 'MCC': 0.40}

## 4. Labels and metadata

In [ ]:
import numpy as np

y = np.load(Y_PATH)
meta = np.load(META_PATH, allow_pickle=True)
N = len(y)
groups = np.asarray(meta['accession']).astype(str)
ALL_PROTEINS = np.unique(groups)

print(f'{N} windows, {int((y==1).sum())} positive, {int((y==0).sum())} negative')
print(f'{len(ALL_PROTEINS)} proteins, imbalance '
      f'{(y==0).sum()/(y==1).sum():.1f}:1')
assert len(groups) == N

252428 windows, 23441 positive, 228987 negative
5606 proteins, imbalance 9.8:1


## 5. Feature array on local disk

Not loaded into RAM this time. Per-epoch resampling means the train bin draws
from the whole negative pool, which would be ~9 GB in memory, so the training
set is read from the local memmap instead. Validation and test are small enough
to hold in RAM and are read every epoch, so that is where it pays.

In [ ]:
import shutil, time

LOCAL = '/content/acet25_X.npy'
if not os.path.exists(LOCAL):
    t = time.time()
    print(f'copying {os.path.getsize(X_PATH)/1e9:.1f} GB to local disk...')
    shutil.copy(X_PATH, LOCAL)
    print(f'done in {(time.time()-t)/60:.1f} min')

X = np.load(LOCAL, mmap_mode='r')
print('X:', X.shape, X.dtype)
assert len(X) == N and X.shape[1] == SEQ_LEN

copying 12.9 GB to local disk...
done in 3.0 min
X: (252428, 25, 1024) float16


## 6. Split by protein

`build_split` now returns the **full** train positive and negative row lists,
not just one undersampled draw, so the training loop can redraw each epoch.
Validation and test are undersampled once and then frozen.

In [ ]:
BINS = ('train', 'val', 'test')


def build_split(seed, verbose=True):
    rng = np.random.default_rng(seed)

    prots = np.array(ALL_PROTEINS, copy=True)
    rng.shuffle(prots)
    n_test = int(round(len(prots) * TEST_FRAC))
    n_val  = int(round(len(prots) * VAL_FRAC))
    parts = {'test':  prots[:n_test],
             'val':   prots[n_test:n_test + n_val],
             'train': prots[n_test + n_val:]}

    pairs = [('train','val'), ('train','test'), ('val','test')]
    for a, b in pairs:
        assert not (set(parts[a]) & set(parts[b])), f'protein leakage {a}/{b}'

    pools, sel, counts = {}, {}, {}
    for name in BINS:
        mask = np.isin(groups, parts[name])
        pos = np.where(mask & (y == 1))[0]
        neg = np.where(mask & (y == 0))[0]
        n = min(len(pos), len(neg))
        assert n > 0
        pools[name] = (pos, neg)
        sel[name] = np.sort(np.concatenate(
            [rng.choice(pos, n, replace=False),
             rng.choice(neg, n, replace=False)]))
        counts[f'avail_{name}_pos'] = int(len(pos))
        counts[f'avail_{name}_neg'] = int(len(neg))
        counts[f'kept_{name}_pos']  = int(n)
        counts[f'kept_{name}_neg']  = int(n)

    used = {k: set(groups[sel[k]].tolist()) for k in BINS}
    for a, b in pairs:
        assert not (used[a] & used[b]), f'leakage {a}/{b} after sampling'

    if verbose:
        print('  ' + '  '.join(f'{b}: {len(parts[b])} proteins, '
              f'{counts[f"kept_{b}_pos"]}+{counts[f"kept_{b}_pos"]} windows'
              for b in BINS))
        pos_tr, neg_tr = pools['train']
        if RESAMPLE_NEG:
            print(f'  per-epoch resampling: {len(neg_tr)} train negatives '
                  f'available, {len(pos_tr)} drawn per epoch '
                  f'({100*len(pos_tr)/len(neg_tr):.1f}% each epoch, '
                  f'most of the pool over {EPOCHS} epochs)')

    return sel, counts, pools, used

## 7. Model and datasets

The architecture is byte-for-byte the one you have been running. Only `DROPOUT`
differs, and it comes from the config cell.

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class MemmapDataset(Dataset):
    """Reads rows from the local memmap. `rows` is swapped each epoch."""
    def __init__(self, rows):
        self.rows = np.asarray(rows)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = int(self.rows[i])
        return (torch.from_numpy(np.asarray(X[r], dtype=np.float32)),
                torch.tensor(float(y[r])))


class RamDataset(Dataset):
    """Fixed bins held in RAM, since they are evaluated every epoch."""
    def __init__(self, rows):
        self.x = np.empty((len(rows), SEQ_LEN, X.shape[2]), dtype=np.float16)
        step = 4096
        for i in range(0, len(rows), step):
            self.x[i:i+step] = X[rows[i:i+step]]
        self.y = y[rows].astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return (torch.from_numpy(self.x[i].astype(np.float32)),
                torch.tensor(self.y[i]))


class LearnablePositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)

    def forward(self, x):
        return x + self.pos_embedding


class ProteinTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.input_norm = nn.LayerNorm(HIDDEN_DIM)
        self.pos_encoding = LearnablePositionalEncoding(SEQ_LEN, HIDDEN_DIM)
        layer = nn.TransformerEncoderLayer(
            d_model=HIDDEN_DIM, nhead=NHEAD, dim_feedforward=FF_DIM,
            dropout=DROPOUT, batch_first=True, activation='gelu',
            norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)
        self.out_norm = nn.LayerNorm(HIDDEN_DIM)
        self.classifier = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, 256), nn.ReLU(),
            nn.Dropout(DROPOUT), nn.Linear(256, 1))

    def forward(self, x):
        x = self.pos_encoding(self.input_norm(self.proj(x)))
        x = self.out_norm(self.transformer(x))
        c = x.shape[1] // 2
        return self.classifier(
            torch.cat([x[:, c, :], x.mean(dim=1)], dim=1)).squeeze(-1)


print(f'{sum(p.numel() for p in ProteinTransformer().parameters())/1e6:.2f}M parameters')

7.11M parameters


/tmp/ipykernel_434/1442657635.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)


## 8. Training with per-epoch resampling

In [ ]:
import gc, json, math
from sklearn.metrics import (confusion_matrix, matthews_corrcoef,
                             accuracy_score, roc_auc_score,
                             average_precision_score)


@torch.no_grad()
def evaluate_probs(model, loader):
    model.eval()
    L, P = [], []
    for xb, yb in loader:
        P.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
        L.append(yb.numpy())
    return np.concatenate(L), np.concatenate(P)


def metrics(labels, probs, thresh=THRESHOLD):
    preds = (probs > thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sn = tp / (tp + fn) if (tp + fn) else 0.0
    sp = tn / (tn + fp) if (tn + fp) else 0.0
    return {'Sn': sn, 'Sp': sp, 'gap': abs(sn - sp),
            'Acc': accuracy_score(labels, preds),
            'MCC': matthews_corrcoef(labels, preds),
            'AUC': roc_auc_score(labels, probs),
            'AUPRC': average_precision_score(labels, probs),
            'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}


def make_scheduler(opt):
    """Linear warmup then cosine decay to zero."""
    def f(ep):
        if ep < WARMUP:
            return (ep + 1) / WARMUP
        t = (ep - WARMUP) / max(1, EPOCHS - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * t))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


def train_one_model(pools, val_loader, seed, member):
    """Train a single network; return its validation and test probabilities."""
    torch.manual_seed(seed * 100 + member)
    torch.cuda.manual_seed_all(seed * 100 + member)
    rng = np.random.default_rng(seed * 100 + member)

    pos_tr, neg_tr = pools['train']
    n = len(pos_tr)

    model = ProteinTransformer().to(device)
    criterion = nn.BCEWithLogitsLoss()      # no pos_weight: 1:1 every epoch
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = make_scheduler(opt)

    best_score, best_state, counter = -2.0, None, 0
    for epoch in range(EPOCHS):
        # redraw a fresh balanced negative sample for this epoch
        if RESAMPLE_NEG:
            rows = np.concatenate([pos_tr,
                                   rng.choice(neg_tr, n, replace=False)])
        else:
            rows = np.concatenate([pos_tr,
                                   rng.choice(neg_tr, n, replace=False)]) \
                   if epoch == 0 else rows
        rng.shuffle(rows)
        loader = DataLoader(MemmapDataset(rows), batch_size=BATCH,
                            shuffle=True, drop_last=True,
                            num_workers=2, pin_memory=True)

        model.train()
        total = 0.0
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item()
        sched.step()

        vl, vp = evaluate_probs(model, val_loader)
        vm = metrics(vl, vp)
        score = min(vm['Sn'], vm['Sp']) if SELECT_BY == 'min_sn_sp' else vm['MCC']

        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            print(f'    ep {epoch+1:3d}/{EPOCHS} loss {total/len(loader):.4f}'
                  f' | val Sn {vm["Sn"]:.3f} Sp {vm["Sp"]:.3f}'
                  f' MCC {vm["MCC"]:.4f} AUC {vm["AUC"]:.4f}'
                  f' | lr {sched.get_last_lr()[0]:.2e}')

        if score > best_score:
            best_score, counter = score, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            counter += 1
            if counter >= PATIENCE:
                print(f'    early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    print(f'    member {member}: best val {SELECT_BY} {best_score:.4f}')
    return model


def train_one_split(seed):
    sel, counts, pools, used = build_split(seed)

    with open(os.path.join(OUTDIR, f'split_proteins_seed{seed}.json'), 'w') as f:
        json.dump({k: sorted(used[k]) for k in BINS}, f)

    val_loader  = DataLoader(RamDataset(sel['val']),  batch_size=512)
    test_loader = DataLoader(RamDataset(sel['test']), batch_size=512)

    test_probs = []
    for member in range(ENSEMBLE):
        print(f'  --- member {member+1}/{ENSEMBLE} ---')
        model = train_one_model(pools, val_loader, seed, member)
        tl, tp_ = evaluate_probs(model, test_loader)
        test_probs.append(tp_)
        single = metrics(tl, tp_)
        print(f'    test alone: Sn {single["Sn"]:.4f} Sp {single["Sp"]:.4f} '
              f'MCC {single["MCC"]:.4f} AUC {single["AUC"]:.4f}')
        torch.save(model.state_dict(),
                   os.path.join(OUTDIR, f'model_seed{seed}_m{member}.pt'))
        del model
        gc.collect(); torch.cuda.empty_cache()

    probs = np.mean(test_probs, axis=0)
    met = metrics(tl, probs)

    print(f'\n  ENSEMBLE of {ENSEMBLE} — seed {seed}, threshold {THRESHOLD}')
    print(f'    confusion   pred neg   pred pos')
    print(f'    actual neg  {met["TN"]:8d}   {met["FP"]:8d}')
    print(f'    actual pos  {met["FN"]:8d}   {met["TP"]:8d}')
    flags = ''.join('  [target met]' if met[k] >= v else '  [below target]'
                    for k, v in [('MCC', 0.40)])
    for k in KEYS:
        t = TARGET.get(k)
        tag = '' if t is None else ('  <- target met' if met[k] >= t
                                    else f'  <- below {t}')
        print(f'    {k:6s} {met[k]:.4f}{tag}')

    np.savez(os.path.join(OUTDIR, f'test_predictions_seed{seed}.npz'),
             y=tl, probs=probs, rows=sel['test'], threshold=THRESHOLD)
    return met, counts

## 9. Run

Resumable: a seed whose results file already exists on Drive is skipped, so
reconnect after a Colab timeout, rerun cells 1-8, and rerun this cell.

In [ ]:
import csv

t_start = time.time()
ran = []
for i, seed in enumerate(SPLIT_SEEDS, 1):
    path = os.path.join(OUTDIR, f'results_seed{seed}.csv')
    if os.path.exists(path):
        print(f'[{i}/{len(SPLIT_SEEDS)}] seed {seed}: already on Drive, skipping')
        continue

    print('\n' + '=' * 70)
    print(f'[{i}/{len(SPLIT_SEEDS)}]  SEED {seed}')
    print('=' * 70)
    t0 = time.time()
    met, counts = train_one_split(seed)
    mins = (time.time() - t0) / 60
    ran.append(mins)

    row = {'split_seed': seed, 'threshold': THRESHOLD,
           **{k: met[k] for k in (*KEYS, *CM_KEYS)}, **counts,
           'ensemble': ENSEMBLE, 'select_by': SELECT_BY,
           'resample_neg': int(RESAMPLE_NEG), 'minutes': round(mins, 2)}
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerow(row)

    left = len(SPLIT_SEEDS) - i
    print(f'\n  wrote {path}')
    print(f'  {mins:.1f} min. {left} left, about '
          f'{sum(ran)/len(ran)*left/60:.1f} h to go.')
    gc.collect(); torch.cuda.empty_cache()

print(f'\nsession time {(time.time()-t_start)/60:.1f} min')


[1/10]  SEED 42
  train: 3644 proteins, 14971+14971 windows  val: 841 proteins, 3462+3462 windows  test: 1121 proteins, 5008+5008 windows
  per-epoch resampling: 146361 train negatives available, 14971 drawn per epoch (10.2% each epoch, most of the pool over 80 epochs)
  --- member 1/3 ---


/tmp/ipykernel_434/1442657635.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)


    ep   1/80 loss 0.6072 | val Sn 0.668 Sp 0.696 MCC 0.3644 AUC 0.7441 | lr 1.20e-04
    ep   6/80 loss 0.4745 | val Sn 0.607 Sp 0.769 MCC 0.3814 AUC 0.7599 | lr 3.00e-04
    ep  11/80 loss 0.2867 | val Sn 0.475 Sp 0.832 MCC 0.3283 AUC 0.7283 | lr 2.95e-04
    ep  16/80 loss 0.2034 | val Sn 0.417 Sp 0.867 MCC 0.3175 AUC 0.7299 | lr 2.84e-04
    early stop at epoch 17
    member 0: best val min_sn_sp 0.6823
    test alone: Sn 0.6667 Sp 0.7013 MCC 0.3682 AUC 0.7469
  --- member 2/3 ---


/tmp/ipykernel_434/1442657635.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)


    ep   1/80 loss 0.6066 | val Sn 0.635 Sp 0.743 MCC 0.3800 AUC 0.7434 | lr 1.20e-04
    ep   6/80 loss 0.4722 | val Sn 0.683 Sp 0.694 MCC 0.3773 AUC 0.7503 | lr 3.00e-04
    ep  11/80 loss 0.2825 | val Sn 0.461 Sp 0.848 MCC 0.3352 AUC 0.7308 | lr 2.95e-04
    ep  16/80 loss 0.1967 | val Sn 0.409 Sp 0.875 MCC 0.3212 AUC 0.7260 | lr 2.84e-04
    ep  21/80 loss 0.1617 | val Sn 0.408 Sp 0.882 MCC 0.3296 AUC 0.7185 | lr 2.68e-04
    early stop at epoch 21
    member 1: best val min_sn_sp 0.6828
    test alone: Sn 0.6807 Sp 0.6999 MCC 0.3807 AUC 0.7534
  --- member 3/3 ---


...

    ep   1/80 loss 0.6117 | val Sn 0.708 Sp 0.657 MCC 0.3651 AUC 0.7479 | lr 1.20e-04
    ep   6/80 loss 0.4806 | val Sn 0.595 Sp 0.780 MCC 0.3807 AUC 0.7554 | lr 3.00e-04
    ep  11/80 loss 0.2971 | val Sn 0.505 Sp 0.803 MCC 0.3228 AUC 0.7129 | lr 2.95e-04
    ep  16/80 loss 0.2143 | val Sn 0.500 Sp 0.818 MCC 0.3356 AUC 0.7260 | lr 2.84e-04
    early stop at epoch 18
    member 0: best val min_sn_sp 0.6659
    test alone: Sn 0.6740 Sp 0.7329 MCC 0.4077 AUC 0.7765
  --- member 2/3 ---


/tmp/ipykernel_434/1442657635.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)


    ep   1/80 loss 0.6077 | val Sn 0.725 Sp 0.638 MCC 0.3644 AUC 0.7470 | lr 1.20e-04
    ep   6/80 loss 0.4820 | val Sn 0.533 Sp 0.811 MCC 0.3588 AUC 0.7445 | lr 3.00e-04
    ep  11/80 loss 0.2882 | val Sn 0.530 Sp 0.813 MCC 0.3575 AUC 0.7383 | lr 2.95e-04
    ep  16/80 loss 0.2169 | val Sn 0.519 Sp 0.808 MCC 0.3417 AUC 0.7215 | lr 2.84e-04
    early stop at epoch 19
    member 1: best val min_sn_sp 0.6916
    test alone: Sn 0.6922 Sp 0.7102 MCC 0.4024 AUC 0.7722
  --- member 3/3 ---


/tmp/ipykernel_434/1442657635.py:57: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(layer, num_layers=NLAYERS)


    ep   1/80 loss 0.6099 | val Sn 0.701 Sp 0.646 MCC 0.3481 AUC 0.7407 | lr 1.20e-04
    ep   6/80 loss 0.4778 | val Sn 0.728 Sp 0.638 MCC 0.3669 AUC 0.7565 | lr 3.00e-04
    ep  11/80 loss 0.2961 | val Sn 0.467 Sp 0.846 MCC 0.3386 AUC 0.7232 | lr 2.95e-04
    ep  16/80 loss 0.2216 | val Sn 0.526 Sp 0.811 MCC 0.3516 AUC 0.7376 | lr 2.84e-04
    early stop at epoch 20
    member 2: best val min_sn_sp 0.6738
    test alone: Sn 0.6632 Sp 0.7244 MCC 0.3883 AUC 0.7643

  ENSEMBLE of 3 — seed 951, threshold 0.5
    confusion   pred neg   pred pos
    actual neg      3532       1257
    actual pos      1500       3289
    Sn     0.6868  <- below 0.7
    Sp     0.7375  <- target met
    gap    0.0507
    Acc    0.7122
    MCC    0.4249  <- target met
    AUC    0.7880
    AUPRC  0.7880

  wrote /content/drive/MyDrive/dbPTM_Acetylation/results_colab_v2/results_seed951.csv
  4.5 min. 0 left, about 0.0 h to go.

session time 44.9 min


## 10. Summary and target check

In [ ]:
import glob
import pandas as pd

rows = []
for p in sorted(glob.glob(os.path.join(OUTDIR, 'results_seed*.csv'))):
    with open(p, newline='') as f:
        rows.extend(list(csv.DictReader(f)))
df = pd.DataFrame(rows)
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors='ignore')
df = df.sort_values('split_seed').reset_index(drop=True)

print(f'{len(df)} of {len(SPLIT_SEEDS)} seeds\n')
print(df[['split_seed', *KEYS]].to_string(index=False, float_format='%.4f'))

if len(df) > 1:
    print('\nreport as: ' + ',  '.join(
        f'{k} {df[k].mean():.4f} +/- {df[k].std(ddof=1):.4f}' for k in KEYS))

print('\ntarget: Sn >= 0.70, Sp >= 0.70, MCC >= 0.40')
for k, t in TARGET.items():
    m, hit = df[k].mean(), int((df[k] >= t).sum())
    print(f'  {k:4s} mean {m:.4f} ({"PASS" if m >= t else "below"}), '
          f'{hit}/{len(df)} seeds individually at or above {t}')

print('\nceiling check: on balanced data MCC = Sn + Sp - 1, so balanced')
print('accuracy is (MCC+1)/2. AUC must rise for both Sn and Sp to clear 0.70.')
print(df[['split_seed', 'AUC', 'MCC', 'Sn', 'Sp', 'gap']].to_string(
      index=False, float_format='%.4f'))

out = os.path.join(OUTDIR, 'results_all_seeds.csv')
df.to_csv(out, index=False)
print(f'\nwrote {out}')
df

10 of 10 seeds

 split_seed     Sn     Sp    gap    Acc    MCC    AUC  AUPRC
         42 0.6915 0.7190 0.0276 0.7053 0.4107 0.7717 0.7789
        143 0.6652 0.7166 0.0514 0.6909 0.3823 0.7571 0.7612
        244 0.6584 0.7429 0.0845 0.7007 0.4028 0.7731 0.7770
        345 0.7061 0.6972 0.0089 0.7017 0.4034 0.7729 0.7740
        446 0.6697 0.7231 0.0533 0.6964 0.3934 0.7682 0.7674
        547 0.6846 0.7414 0.0567 0.7130 0.4267 0.7845 0.7887
        648 0.7028 0.7028 0.0000 0.7028 0.4055 0.7720 0.7730
        749 0.6840 0.7319 0.0479 0.7079 0.4163 0.7759 0.7772
        850 0.6868 0.7133 0.0264 0.7001 0.4003 0.7687 0.7684
        951 0.6868 0.7375 0.0507 0.7122 0.4249 0.7880 0.7880

report as: Sn 0.6836 +/- 0.0153,  Sp 0.7226 +/- 0.0158,  gap 0.0407 +/- 0.0250,  Acc 0.7031 +/- 0.0068,  MCC 0.4066 +/- 0.0137,  AUC 0.7732 +/- 0.0086,  AUPRC 0.7754 +/- 0.0086

target: Sn >= 0.70, Sp >= 0.70, MCC >= 0.40
  Sn   mean 0.6836 (below), 2/10 seeds individually at or above 0.7
  Sp   mean 0.7226 (PA

/tmp/ipykernel_434/278619705.py:10: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[c] = pd.to_numeric(df[c], errors='ignore')


,split_seed,threshold,Sn,Sp,gap,Acc,MCC,AUC,AUPRC,TN,...,kept_val_pos,kept_val_neg,avail_test_pos,avail_test_neg,kept_test_pos,kept_test_neg,ensemble,select_by,resample_neg,minutes
0,42,0.5,0.691494,0.719050,0.027556,0.705272,0.410699,0.771727,0.778870,3601,...,3462,3462,5008,47585,5008,5008,3,min_sn_sp,1,4.75
1,143,0.5,0.665216,0.716594,0.051378,0.690905,0.382316,0.757079,0.761240,3459,...,3354,3354,4827,45451,4827,4827,3,min_sn_sp,1,4.14
2,244,0.5,0.658448,0.742925,0.084477,0.700686,0.402812,0.773145,0.776969,3465,...,3608,3608,4664,46288,4664,4664,3,min_sn_sp,1,4.28
3,345,0.5,0.706095,0.697244,0.008851,0.701670,0.403355,0.772937,0.774036,3466,...,3154,3154,4971,46495,4971,4971,3,min_sn_sp,1,4.46
4,446,0.5,0.669746,0.723074,0.053328,0.696410,0.393379,0.768194,0.767401,3444,...,3666,3666,4763,46718,4763,4763,3,min_sn_sp,1,4.57
5,547,0.5,0.684650,0.741383,0.056733,0.713016,0.426720,0.784488,0.788679,3463,...,3759,3759,4671,46165,4671,4671,3,min_sn_sp,1,4.84
6,648,0.5,0.702761,0.702761,0.000000,0.702761,0.405523,0.771976,0.772970,3232,...,3500,3500,4599,46849,4599,4599,3,min_sn_sp,1,4.41
7,749,0.5,0.683994,0.731861,0.047867,0.707927,0.416332,0.775868,0.777156,3379,...,3368,3368,4617,47613,4617,4617,3,min_sn_sp,1,4.57
8,850,0.5,0.686844,0.713287,0.026442,0.700066,0.400271,0.768652,0.768357,3264,...,3219,3219,4576,48959,4576,4576,3,min_sn_sp,1,4.40
9,951,0.5,0.686782,0.737523,0.050741,0.712153,0.424853,0.788018,0.787952,3532,...,3697,3697,4789,44394,4789,4789,3,min_sn_sp,1,4.50


## 11. If it still falls short

Run one seed with the knobs below before committing to all ten.

- `ENSEMBLE = 5` — another 0.005-0.01 AUC, at 5/3 the time.
- `LR = 5e-4`, `BATCH = 256` — if the loss curve is still falling when early
  stopping fires, the run is still undertrained.
- `NLAYERS = 4`, `HIDDEN_DIM = 640` — more capacity now that per-epoch
  resampling supplies far more distinct training data. Departs from the frozen
  architecture, so report it separately.
- `DROPOUT = 0.1`.

If AUC stays near 0.76 through all of that, the ceiling is in the features, not
the training. A 25-residue window carries limited context; widening to 33 or 41
residues, or adding the full-protein mean embedding as a side input, is the
change that moves AUC. Those need re-extraction, not retraining.